In [ ]:
%cd ../..
import os
import polars as pl
import numpy as np
import random
from typing import OrderedDict
from glob import glob

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/scratch/VM/radio-foundation/cache/embeddings/CCCCII"

class_names = ["CP", "NCP", "Normal"]

class_ids = {}
for c in class_names:
    class_files = os.listdir(os.path.join(embeddings_path, c))
    class_ids[c] = [f.replace(".pth", "") for f in class_files]

all_paths = glob(os.path.join(embeddings_path, "**/*.pth"), recursive=True)

id_to_path = {
    path.split("/")[-1].replace(".pth", "") : path for path in all_paths
}

In [ ]:
class_ids["CP"]

In [ ]:
# CP vs NCP
labels = OrderedDict()
labels.update({p_id: 0 for p_id in class_ids["CP"]})
labels.update({p_id: 1 for p_id in class_ids["NCP"]})
patient_ids = list(labels.keys())
labels_list = list(labels.values())
num_classes = 2
len(patient_ids)

In [ ]:
# CP or NCP vs Normal
labels = OrderedDict()
labels.update({p_id: 0 for p_id in class_ids["Normal"]})
labels.update({p_id: 1 for p_id in class_ids["CP"]})
labels.update({p_id: 1 for p_id in class_ids["NCP"]})
patient_ids = list(labels.keys())
labels_list = list(labels.values())
num_classes = 2
len(patient_ids)

In [ ]:
train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=5, stratify=labels_list)

train_dataset = EmbeddingDataset(train_ids, id_to_path, labels, add_noise=True, p=1.0, sigma=0.1)
val_dataset = EmbeddingDataset(val_ids, id_to_path, labels)

In [ ]:
class_weights = get_class_weights(labels_list)
class_weights

In [ ]:
EMBED_DIM = 768
num_epochs = 50
batch_size = 16
learning_rate = 0.001

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=collate_classification,
    batch_size=batch_size,
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_classification)

device = torch.device("cuda")
pooler = GatedPool(EMBED_DIM)
model = Classifier(pooler, embed_dim=EMBED_DIM, num_classes=num_classes).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

output = train_classifier(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(output["state_dict"])


In [ ]:
plot_train_curves(output["train_loss"], output["val_loss"], "Cross-Entropy Loss")

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device)
all_predictions = torch.nn.functional.softmax(all_predictions, dim=1)
all_predictions = torch.argmax(all_predictions, dim=1)

plot_confusion_matrix(all_labels, all_predictions)